In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType,
)
from pyspark.sql import functions as F

In [2]:
# spark = (
#     SparkSession.builder
#     .appName("Lakehouse Implementation")
#     .master("spark://spark-master:7077")

#     .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
#     # Make Iceberg the default table format for spark_catalog (Hive Metastore backed)
#     .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkSessionCatalog")
#     .config("spark.sql.catalog.spark_catalog.type", "hive")
#     .config("spark.sql.catalog.spark_catalog.uri", "thrift://hive-metastore:9083")
#     .config("spark.sql.catalogImplementation", "hive")
#     .config("spark.hadoop.hive.metastore.uris", "thrift://hive-metastore:9083")      # Hive Metastore connection
#     .enableHiveSupport()

#     # MinIO / S3A
#     .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
#     .config("spark.hadoop.fs.s3a.access.key", "admin")
#     .config("spark.hadoop.fs.s3a.secret.key", "12345678")
#     .config("spark.hadoop.fs.s3a.path.style.access", "true")
#     .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
#     .config("spark.sql.warehouse.dir", "s3a://buck1/warehouse")   # warehouse location

#     .getOrCreate()
# )
import os
CATALOG = "lakehouse"
C = f"spark.sql.catalog.{CATALOG}"

spark = (
    SparkSession.builder
    .appName("Lakehouse Implementation")
    .master("spark://spark-master:7077")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")

    # Named Iceberg catalog backed by the Hive Metastore
    .config(C, "org.apache.iceberg.spark.SparkCatalog")
    .config(f"{C}.type", "hive")
    .config(f"{C}.uri", "thrift://hive-metastore:9083")
    .config(f"{C}.warehouse", "s3://buck1/warehouse")

    # Iceberg's own S3 client for table data and metadata.
    # Keys and region come from AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY / AWS_REGION.
    .config(f"{C}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"{C}.s3.endpoint", os.environ["S3_ENDPOINT"])
    .config(f"{C}.s3.path-style-access", "true")
    .config(f"{C}.http-client.type", "apache")   # the only HTTP client in bundle-2.29.52.jar

    .config("spark.sql.defaultCatalog", CATALOG)

    # S3A is still used for streaming checkpoints (Spark writes them through Hadoop, not Iceberg).
    # S3A's default credential chain reads the same AWS_* environment variables.
    .config("spark.hadoop.fs.s3a.endpoint", os.environ["S3_ENDPOINT"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 07:43:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.sql("SELECT current_catalog()").show()

+-----------------+
|current_catalog()|
+-----------------+
|        lakehouse|
+-----------------+



In [4]:
print(
    spark.sql(
        "SHOW DATABASES"
    ).show()
)

+---------+
|namespace|
+---------+
|  default|
+---------+

None


In [4]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS demo
""")

DataFrame[]

26/08/14 15:52:05 ERROR TaskSchedulerImpl: Lost executor 9 on 172.18.0.9: worker lost: Not receiving heartbeat for 60 seconds
26/08/14 15:52:05 ERROR TaskSchedulerImpl: Lost executor 8 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/08/14 16:19:37 ERROR TaskSchedulerImpl: Lost executor 11 on 172.18.0.9: worker lost: Not receiving heartbeat for 60 seconds
26/08/14 16:19:37 ERROR TaskSchedulerImpl: Lost executor 10 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds


In [4]:
spark.sql("""
SHOW Tables in demo
""").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [6]:
# Use the demo database
spark.sql("USE demo")

# ---------- Table 1: customers ----------
spark.sql("""
CREATE TABLE IF NOT EXISTS demo.customers (
    customer_id INT,
    name STRING,
    city STRING
)
USING PARQUET
""")

spark.sql("""
INSERT INTO demo.customers VALUES
    (1, 'Ali Raza', 'Lahore'),
    (2, 'Sara Khan', 'Karachi'),
    (3, 'Bilal Ahmed', 'Islamabad'),
    (4, 'Mahnoor Iqbal', 'Lahore')
""")

# ---------- Table 2: orders ----------
spark.sql("""
CREATE TABLE IF NOT EXISTS demo.orders (
    order_id INT,
    customer_id INT,
    amount DOUBLE,
    order_date STRING
)
USING PARQUET
""")

spark.sql("""
INSERT INTO demo.orders VALUES
    (101, 1, 250.50, '2026-01-05'),
    (102, 2, 89.99,  '2026-01-06'),
    (103, 1, 430.00, '2026-01-10'),
    (104, 3, 12.75,  '2026-02-01'),
    (105, 4, 999.00, '2026-02-03')
""")

# ---------- Query 1: simple select ----------
spark.sql("SELECT * FROM demo.customers").show()

# ---------- Query 2: join + aggregation ----------
spark.sql("""
SELECT c.name, c.city, COUNT(o.order_id) AS num_orders, SUM(o.amount) AS total_spent
FROM demo.customers c
JOIN demo.orders o ON c.customer_id = o.customer_id
GROUP BY c.name, c.city
ORDER BY total_spent DESC
""").show()

26/08/11 12:46:05 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


+-----------+-------------+---------+
|customer_id|         name|     city|
+-----------+-------------+---------+
|          3|  Bilal Ahmed|Islamabad|
|          4|Mahnoor Iqbal|   Lahore|
|          2|    Sara Khan|  Karachi|
|          1|     Ali Raza|   Lahore|
+-----------+-------------+---------+



+-------------+---------+----------+-----------+
|         name|     city|num_orders|total_spent|
+-------------+---------+----------+-----------+
|Mahnoor Iqbal|   Lahore|         1|      999.0|
|     Ali Raza|   Lahore|         2|      680.5|
|    Sara Khan|  Karachi|         1|      89.99|
|  Bilal Ahmed|Islamabad|         1|      12.75|
+-------------+---------+----------+-----------+



In [7]:
print(spark.conf.get("spark.sql.catalogImplementation"))

hive


In [8]:
spark.sql("DESCRIBE FORMATTED demo.customers").show(100, truncate=False)

+----------------------------+--------------------------------------------------------------+-------+
|col_name                    |data_type                                                     |comment|
+----------------------------+--------------------------------------------------------------+-------+
|customer_id                 |int                                                           |NULL   |
|name                        |string                                                        |NULL   |
|city                        |string                                                        |NULL   |
|                            |                                                              |       |
|# Detailed Table Information|                                                              |       |
|Catalog                     |spark_catalog                                                 |       |
|Database                    |demo                                                

In [ ]:
# currently tables are plain Parquet + Hive Metastore, no Iceberg yet.

In [6]:
spark.sql("USE demo")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.customers_iceberg (
    customer_id INT,
    name STRING,
    city STRING
)
USING ICEBERG
""")

spark.sql("""
INSERT INTO demo.customers_iceberg VALUES
    (1, 'Ali Raza', 'Lahore'),
    (2, 'Sara Khan', 'Karachi'),
    (3, 'Bilal Ahmed', 'Islamabad')
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.orders_iceberg (
    order_id INT,
    customer_id INT,
    amount DOUBLE,
    order_date STRING
)
USING ICEBERG
""")

spark.sql("""
INSERT INTO demo.orders_iceberg VALUES
    (101, 1, 250.50, '2026-01-05'),
    (102, 2, 89.99,  '2026-01-06'),
    (103, 1, 430.00, '2026-01-10')
""")

spark.sql("SELECT * FROM demo.customers_iceberg").show()
spark.sql("""
SELECT c.name, SUM(o.amount) AS total_spent
FROM demo.customers_iceberg c JOIN demo.orders_iceberg o ON c.customer_id = o.customer_id
GROUP BY c.name
""").show()

+-----------+-----------+---------+
|customer_id|       name|     city|
+-----------+-----------+---------+
|          1|   Ali Raza|   Lahore|
|          2|  Sara Khan|  Karachi|
|          3|Bilal Ahmed|Islamabad|
+-----------+-----------+---------+



+---------+-----------+
|     name|total_spent|
+---------+-----------+
| Ali Raza|      680.5|
|Sara Khan|      89.99|
+---------+-----------+



In [4]:
spark.sql("DESCRIBE FORMATTED demo.customers_iceberg").show(100, truncate=False)

+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                             |comment|
+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|customer_id                 |int                                                                                                                   |NULL   |
|name                        |string                                                                                                                |NULL   |
|city                        |string                                                                                                                |NULL   |
|                            |                      

In [12]:
spark.sql("DROP TABLE IF EXISTS demo.customers")
spark.sql("DROP TABLE IF EXISTS demo.orders")

DataFrame[]

In [5]:
spark.sql("""
SHOW Tables in demo
""").show()

+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
|     demo|customers_iceberg|      false|
|     demo|   orders_iceberg|      false|
+---------+-----------------+-----------+



In [14]:
spark.conf.get("spark.sql.warehouse.dir")

's3a://buck1/warehouse'

## Kafka Implementation

In [12]:
kafka_df = spark \
  .readStream \
  .format("kafka") \
  .option("kafka.bootstrap.servers",  "broker:9092") \
  .option("subscribe", "events") \
  .option("startingOffsets", "earliest") \
  .load()

In [13]:
event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("customer_id", IntegerType(), True),
    StructField("event_type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("event_time", StringType(), True),
])

In [14]:
events_df = (
    kafka_df
    .select(
        F.from_json(
            F.col("value").cast("string"),
            event_schema
        ).alias("data")
    )
    .select("data.*")
    .withColumn(
        "event_time",
        F.to_timestamp("event_time")
    )
)

In [15]:
query = (
    events_df
    .writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation","s3a://buck1/checkpoints/events-v1")
    .trigger(processingTime="10 seconds")
    .toTable(
        "spark_catalog.demo.events"
    )
)
# query.awaitTermination()

26/08/15 07:11:57 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [16]:
spark.sql("""
SELECT * FROM demo.events
""").show()

+--------------------+-----------+----------+------+--------------------+
|            event_id|customer_id|event_type|amount|          event_time|
+--------------------+-----------+----------+------+--------------------+
|6ad861c5-329d-440...|        974|     login|433.74|2026-08-15 06:35:...|
|1d64abdc-a4fe-4ba...|        187| page_view|273.26|2026-08-15 06:35:...|
|869d91f2-b9c3-415...|        573|  purchase|121.67|2026-08-15 06:35:...|
|27d1e5c5-1337-453...|        452|    logout|158.29|2026-08-15 06:35:...|
|592e6c1c-2b7d-4db...|        675|     login| 11.33|2026-08-15 06:35:...|
|7245362a-6d49-4b2...|        471|    logout|117.04|2026-08-15 06:35:...|
|7d54c8a4-3217-474...|        369|     login|244.58|2026-08-15 06:35:...|
|5e9178ed-8c98-4d8...|        576| page_view|180.79|2026-08-15 06:35:...|
|fbc6db89-56fa-40a...|        256|    logout|220.49|2026-08-15 06:35:...|
|39c6a997-efc9-4b2...|        493| page_view|251.72|2026-08-15 06:35:...|
|1d021973-ada5-472...|         28|    

In [17]:
spark.sql("DESCRIBE FORMATTED demo.events").show(100, truncate=False)

+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                             |comment|
+----------------------------+----------------------------------------------------------------------------------------------------------------------+-------+
|event_id                    |string                                                                                                                |NULL   |
|customer_id                 |int                                                                                                                   |NULL   |
|event_type                  |string                                                                                                                |NULL   |
|amount                      |double                

26/08/15 07:29:21 ERROR TaskSchedulerImpl: Lost executor 1 on 172.18.0.7: worker lost: Not receiving heartbeat for 60 seconds
26/08/15 07:29:21 ERROR TaskSchedulerImpl: Lost executor 0 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/08/15 07:31:20 ERROR TaskSchedulerImpl: Lost executor 3 on 172.18.0.7: worker lost: Not receiving heartbeat for 60 seconds
26/08/15 07:31:20 ERROR TaskSchedulerImpl: Lost executor 2 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/08/15 08:15:05 ERROR TaskSchedulerImpl: Lost executor 4 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/08/15 08:15:05 ERROR TaskSchedulerImpl: Lost executor 5 on 172.18.0.7: worker lost: Not receiving heartbeat for 60 seconds


In [18]:
query.stop()

26/08/15 09:03:41 WARN DAGScheduler: Failed to cancel job group 67df735c-2417-42be-bf16-c30c80d496ae. Cannot find active jobs for it.
26/08/15 09:03:41 WARN DAGScheduler: Failed to cancel job group 67df735c-2417-42be-bf16-c30c80d496ae. Cannot find active jobs for it.
26/08/15 09:35:11 ERROR TaskSchedulerImpl: Lost executor 6 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/08/15 09:35:11 ERROR TaskSchedulerImpl: Lost executor 7 on 172.18.0.7: worker lost: Not receiving heartbeat for 60 seconds
26/08/15 12:32:21 ERROR TaskSchedulerImpl: Lost executor 9 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/08/15 12:32:21 ERROR TaskSchedulerImpl: Lost executor 8 on 172.18.0.7: worker lost: Not receiving heartbeat for 60 seconds
26/08/15 13:25:11 ERROR TaskSchedulerImpl: Lost executor 10 on 172.18.0.8: worker lost: Not receiving heartbeat for 60 seconds
26/08/15 13:25:11 ERROR TaskSchedulerImpl: Lost executor 11 on 172.18.0.7: worker lost: Not receiving

In [ ]:
# to store stream in kafka
kafka_df.writeStream \
    .format("kafka") \
    .option("checkpointLocation", "/app/checkpoint/") \
    .option("topic", "spark_stream_topic") \
    .option("kafka.bootstrap.servers", "broker:9092") \
    .outputMode("append") \
    .trigger(processingTime="10 seconds") \
    .start() \
    .awaitTermination()

In [38]:
spark.sql("SHOW TABLES IN spark_catalog.demo").show()

+---------+-----------------+-----------+
|namespace|        tableName|isTemporary|
+---------+-----------------+-----------+
|     demo|customers_iceberg|      false|
|     demo|   orders_iceberg|      false|
+---------+-----------------+-----------+



In [3]:
spark.sql("""
CREATE TABLE person (
    id INT,
    name STRING,
    age INT
)
""")

spark.sql("""
INSERT INTO person VALUES
    (100, 'John', 30),
    (200, 'Mary', NULL),
    (300, 'Mike', 80),
    (400, 'Dan', 50),
    (500, 'Evan_w', 16)
""")

IllegalArgumentException: Invalid table identifier: person